# SAM3 — one bands+edge retry with adapted calibration

**Run all after refreshing this notebook.** One model, five training volumes, **25 total optimizer updates**.

1. Updates **1–5:** native SAM3 loss + bands, using the saved bands coefficient.
2. After update **5:** calibrate only edge to a **10% median gradient target**, with the original
   **50% p95 calibration cap**, on the original **20 training-support slabs**. Keep bands fixed.
3. Updates **6–25:** native SAM3 loss + bands + the newly calibrated edge term.
4. Evaluate this final model once on the original **5 training + 96 validation volumes**.

The original learning-rate schedule, global constraint ramp, seeds, sampling and loss definitions
are retained. Calibration and monitoring do not perform optimizer updates. Edge strength is measured
again on five fixed slabs after updates **10 and 25**; these measurements do not change the weights.
The target and cap apply to calibration, not a guaranteed bound throughout training.

Saved baseline, bands and previous bands+edge results are reused; none is retrained or reevaluated.
The completed diagnostic notebook is backed up separately. New results are written to
`results_5vol_25updates_warm5_edge10_v1`. Use the same GPU/software as before.


In [14]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time
NOTEBOOK_STARTED=time.monotonic()
HOME_DIR=Path('/content/drive/MyDrive/hippo_sam3_geofwi_exploration')
INPUT=HOME_DIR/'input'
SOURCE_RESULTS=HOME_DIR/'results_5vol_25updates_bands10_edge10_v1'
PREVIOUS_RESULTS=HOME_DIR/'results_5vol_24conditions_v1'
RESULTS=HOME_DIR/'results_5vol_25updates_warm5_edge10_v1'
BUNDLE=Path('/content/hippo_sam3_geofwi_two_runs_bundle')
BASE_WEIGHTS=Path('/content/drive/MyDrive/hippo_medsam3_msd_extension/input/sam3.pt')
# Change BASE_WEIGHTS if your existing sam3.pt is elsewhere; it is never overwritten.
RESULTS.mkdir(parents=True,exist_ok=True)
print('Results and recovery:',RESULTS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results and recovery: /content/drive/MyDrive/hippo_sam3_geofwi_exploration/results_5vol_25updates_warm5_edge10_v1


## Stage verified inputs on the local SSD
Files already present with matching hashes are reused. Different files cause a stop; active work is preserved.
The data ZIP contains only the exact five training and 96 validation volumes, not the full dataset.


In [15]:
EXPECTED_UPLOADS={'sam3_geofwi_101vol_data.zip': '094456253ed3bf63e792d08b53eeb0c8bd36fc577a857c6cc6557d6f8271c5f0', 'sam3.pt': '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e', 'sam3_geofwi_two_runs_code.zip': '3c7ea0c2f8b95d1dff0bd14d937f7d0238f8379b81ca07dc7fb1643a2b1f4536'}
EXPECTED_MANIFEST='7a8904c0151702fced8c1a922a760efe9f2c7b80afe0f599c1f2cd34b2612684'

import fcntl, shutil, zipfile
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()
def safe(name):
    p=Path(name)
    if p.is_absolute() or '..' in p.parts:raise ValueError('Unsafe ZIP path')
    return p
def install_zip(path,expected,allowed):
    if sha(path)!=expected:raise ValueError('ZIP hash mismatch: '+str(path))
    with zipfile.ZipFile(path) as archive:
        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError('Duplicate ZIP members')
        if set(archive.namelist())!=set(allowed):raise ValueError('ZIP inventory mismatch')
        for info in archive.infolist():
            safe(info.filename)
            if (info.external_attr>>16)&0o170000==0o120000:raise ValueError('Symlink in ZIP')
        for name,expected_sha in allowed.items():
            target=BUNDLE/safe(name)
            if target.exists():
                if target.is_symlink() or sha(target)!=expected_sha:raise ValueError('Existing file differs: '+str(target))
                continue
            target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_name(target.name+'.staging')
            with archive.open(name) as source,tmp.open('wb') as dest:shutil.copyfileobj(source,dest)
            if sha(tmp)!=expected_sha:raise ValueError('Extracted member differs')
            tmp.replace(target)
(BUNDLE/'work').mkdir(parents=True,exist_ok=True)
with (BUNDLE/'work/sweep.lock').open('a') as lock:
    fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    package=INPUT/'sam3_geofwi_two_runs_code.zip'
    if sha(package)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Code ZIP changed')
    with zipfile.ZipFile(package) as archive:
        raw=archive.read('bundle_manifest.json')
        if hashlib.sha256(raw).hexdigest()!=EXPECTED_MANIFEST:raise ValueError('Scientific manifest differs')
        files=json.loads(raw);files['bundle_manifest.json']=EXPECTED_MANIFEST
    install_zip(package,EXPECTED_UPLOADS[package.name],files)
    data_files={'data/'+n:h for n,h in json.loads((BUNDLE/'data_manifest.json').read_text()).items()}
    if any(not (BUNDLE/n).exists() or sha(BUNDLE/n)!=h for n,h in data_files.items()):
        install_zip(INPUT/'sam3_geofwi_101vol_data.zip',EXPECTED_UPLOADS['sam3_geofwi_101vol_data.zip'],data_files)
    base=BUNDLE/'weights/sam3.pt';base.parent.mkdir(exist_ok=True)
    if not base.exists():
        if shutil.disk_usage(BUNDLE).free<16*1024**3:raise RuntimeError('Need16 GiB free runtime SSD before staging')
        if sha(BASE_WEIGHTS)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Existing sam3.pt differs')
        tmp=base.with_suffix('.staging');shutil.copyfile(BASE_WEIGHTS,tmp)
        if sha(tmp)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Checkpoint copy differs')
        tmp.replace(base)
    elif sha(base)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Local SAM3 checkpoint differs')
    provenance=RESULTS/'source'
    provenance.mkdir(exist_ok=True)
    dest=provenance/package.name
    if not dest.exists():shutil.copyfile(package,dest)
    if sha(dest)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Durable source package differs')
print('Verified frozen source,101 volumes and SAM3 weights on local SSD.')


Verified frozen source,101 volumes and SAM3 weights on local SSD.


## Environment and commands
Reuses matching installed versions; otherwise installs the same pinned dependencies as the MSD notebook.
An A100/H100-class BF16 GPU with at least 40 GB VRAM is required. Colab speed and CUDA equivalence
are checked on your actual GPU; local packaging tests cannot establish runtime speedups.


In [16]:
VENV=Path('/content/hippo_sam3_geofwi_venv');PYTHON=VENV/'bin/python'
def setup(args):
    p=subprocess.run(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with (RESULTS/'setup.log').open('a') as log:log.write(p.stdout)
    print('\n'.join(p.stdout.splitlines()[-30:]))
    if p.returncode:raise RuntimeError('Setup failed; see setup.log')
if not PYTHON.exists():setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    import urllib.request
    path=Path('/content/sam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',path)
    setup([PYTHON,path])
probe="import importlib.metadata as m,sys; from pathlib import Path; lines=Path(sys.argv[1]).read_text().splitlines(); ok=all(m.version(s.split('==')[0])==s.split('==')[1] if '==' in s else bool(m.version(s)) for s in lines if s.strip()); sys.exit(0 if ok else 1)"
if subprocess.run([str(PYTHON),'-c',probe,str(BUNDLE/'requirements-colab.txt')],capture_output=True).returncode:
    setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
else:print('Reusing matching dependency versions')
def execute(args,log_name,parallel=False):
    p=subprocess.Popen(list(map(str,args)),start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with (RESULTS/log_name).open('a') as log:
        while True:
            try:
                for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
                code=p.wait();break
            except KeyboardInterrupt:
                print('Clean stop requested. Keep this VM connected until the cell returns and recovery is verified.')
                if p.poll() is None:
                    if parallel:p.send_signal(signal.SIGUSR1)
                    else:os.killpg(p.pid,signal.SIGINT)
    if code:raise RuntimeError('Command failed; see '+str(RESULTS/log_name))
def campaign(command):
    execute([PYTHON,BUNDLE/'runtime.py',command,'--bundle',BUNDLE,'--durable',RESULTS],command+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


Reusing matching dependency versions
Verified 2 GeoFWI3D SAM3 conditions and all101 volumes


CompletedProcess(args=['nvidia-smi'], returncode=0)

## Stage the retry code

Reuse the original frozen bundle and saved references. The runner verifies the original initialization,
training volumes, baseline archive and completed constraint runs before starting.


In [17]:
RETRY_FILES={'sam3_warm_edge_v1.py': 'a3cd0f43b704e88241d3b88317aa8b0ad5a6e4ab5b06c8ad636f753ec0f45f9e', 'sam3_edge_diagnose_v1.py': '2121b3831a71fbd5b17c966a9d701e73a1a2ee066dd1fb5921b1585014bde2ae'}
for name,expected in RETRY_FILES.items():
    source=INPUT/name
    if sha(source)!=expected:raise ValueError('Retry source differs: '+name+'; refresh the notebook')
    destination=Path('/content')/name
    shutil.copyfile(source,destination)
    if sha(destination)!=expected:raise ValueError('Retry copy failed: '+name)
RETRY_SCRIPT=Path('/content/sam3_warm_edge_v1.py')
print('Ready: ONE retry; 5 bands-only updates + 20 bands+edge updates; original controls reused.')


Ready: ONE retry; 5 bands-only updates + 20 bands+edge updates; original controls reused.


## Run or resume the single retry

The checkpoint is saved locally after every update, with periodic verified Drive recovery snapshots
and forced snapshots after calibration and monitoring. Interrupt once and wait for a clean stop
before disconnecting. Rerun this cell to resume. A hard VM loss with unsaved charged updates
raises an error rather than silently repeating them.

Final evaluation is the slow part: only this new model is evaluated. Audited evaluation feature
reuse is enabled with automatic fallback to the original implementation. Completed runs are skipped.
Checkpoints are removed only after the complete metrics archive is verified.


In [18]:
execute([PYTHON,RETRY_SCRIPT,'--bundle',BUNDLE,'--source-results',SOURCE_RESULTS,
         '--previous-results',PREVIOUS_RESULTS,'--durable',RESULTS],'retry.log')


httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Replaced 61 nn.MultiheadAttention modules with MultiheadAttentionLoRA
Applied LoRA to 458 modules:
  - backbone.vision_backbone.trunk.blocks.0.attn.qkv
  - backbone.vision_backbone.trunk.blocks.0.attn.proj
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.1.attn.qkv
  - backbone.vision_backbone.trunk.blocks.1.attn.proj
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.2.attn.qkv
  - backbone.vision_backbone.trunk.blocks.2.attn.proj
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc2
  - backbone.vision_backbone.t

## Results and effective edge strength

The comparison includes the three saved references and this one new run. Dice is measured on
the same validation cases. Gradient ratios use LoRA parameter gradients and are not Adam update shares.
The calibration row summarizes 20 slabs; later monitoring rows summarize five fixed slabs.


In [19]:
import pandas as pd
import numpy as np
comparison=RESULTS/'comparison.json'
if comparison.exists():
    table=pd.DataFrame(json.loads(comparison.read_text())['rows'])
    display(table)
    table.to_csv(RESULTS/'comparison.csv',index=False)
else:
    print('Retry is incomplete; rerun the training cell to resume.')
candidates=list((RESULTS/'runs').glob('*/calibration.json'))
if candidates:
    cal=json.loads(candidates[0].read_text())
    print('Final calibrated coefficients:',cal['weights'])
    groups=[(5,cal['rows'])]+[(int(step),rows) for step,rows in cal['monitoring'].items()]
    rows=[]
    for step,items in sorted(groups):
        full=[r['full_strength']['edge_over_native_bands'] for r in items]
        scheduled=[r['scheduled_strength']['edge_over_native_bands'] for r in items]
        rows.append(dict(after_update=step,slabs=len(items),full_strength_median=float(np.median(full)),
            full_strength_p95=float(np.quantile(full,.95)),full_strength_max=max(full),
            next_update_schedule_median=float(np.median(scheduled)),
            conflicting_slabs=sum(r['full_strength']['cosine_edge_native_bands'] is not None and
                r['full_strength']['cosine_edge_native_bands']<0 for r in items)))
    monitoring=pd.DataFrame(rows);display(monitoring)
    monitoring.to_csv(RESULTS/'edge_monitoring.csv',index=False)
print('Saved results:',RESULTS)


,run,validation_dice,training_dice,foreground_fn,foreground_fp,delta_vs_bands_pp,wins_vs_bands,losses_vs_bands
0,Saved native baseline,0.651022,0.626728,3470411,691717,-3.312317,25,71
1,Saved dice_bands_edge,0.519985,0.556685,4913341,246404,-16.415968,5,91
2,Saved dice_bands,0.684145,0.673578,3242533,561362,0.000000,0,0
3,Adapted-calibration bands+edge,0.625286,0.663745,4082646,360586,-5.885925,20,76


Final calibrated coefficients: {'bands': 5.078932100766348, 'edge': 127.21203011659773}


,after_update,slabs,full_strength_median,full_strength_p95,full_strength_max,next_update_schedule_median,conflicting_slabs
0,5,20,0.019432,0.500000,0.627112,0.005794,6
1,10,5,0.190662,0.441151,0.488450,0.106487,0
2,25,5,0.245830,0.636665,0.688504,0.245830,0


Saved results: /content/drive/MyDrive/hippo_sam3_geofwi_exploration/results_5vol_25updates_warm5_edge10_v1
